# ⚠️ ARCHIVED — PRE-RESET (2026-02-18). DO NOT ACT ON ITS OUTPUTS.

This notebook predates the knowledge-base reset of 2026-07-05 (CONTEXT.md, ADR-0001–0007).
It is kept for its **methods**, not its **results**. Its results are refuted.

## Why the conclusions are void

1. **Its labels come from `ground_truth_offsets.json`**, which is now FROZEN and
   superseded. It uses `tile_formula['base_offset']` = **337,375** — tombstoned as
   `tile-base-337375-grace-anchored`. That number turned out to be real, but it is the
   *distance between two flag families*, not a base; here it is used as a base.
2. **It treats the "dungeon"/"catacombs" regions as flag bitmaps.** They are a u32
   record list. Entries there react to kills, which is why they looked correlated.
3. **It aggregates per-offset counts across 799 snapshots.** This assumes a flag stays
   at a fixed offset over time. It does not: family bases drift *within a single
   session* (docs/BACKLOG.md 4b). So each offset bucket mixes different flags, and
   every feature built on it is smeared.
4. Its "unknown zones" (90K–337K, 1.25M–1.83M) are defined relative to the tombstoned
   base, so those boundaries are meaningless.

## What is still worth having

Several ideas here were re-derived independently later and are now in the pipeline:

| notebook idea | where it ended up |
|---|---|
| single-bit change extraction | isolated-flip analysis, `src/knowledge/pipeline.rs` |
| SET-only / never-cleared behaviour | set-monotonicity, used for cross-checks |
| inventory co-occurrence | `reward_corroboration` (ADR-0007) |
| co-occurrence + union-find grouping | **not yet built** — see below |

The last row is the interesting one. `docs/BACKLOG.md` step 3 names the next viable
timeline design as: *cluster grace-aligned isolated flips across every consecutive pair
in the chain, and locate the family base from where many independent flips agree.*
Cells 27–28 here are a prototype of exactly that. If someone picks that work up, start
by reading them — then rebuild on **pair-relative** offsets rather than absolute ones,
which is the mistake that sinks everything above.

## If you want to revive it

Re-anchor every offset against a resolved origin first:
`wasm_event_flags::resolve_family_base_in_ef` (see `crates/wasm-event-flags/src/lib.rs`).
Take labels from `knowledge/claims/event-flags.json`, never from the frozen store.


# ML-Driven Event Flag Offset Discovery

This notebook explores using machine learning on timeline diff data to discover
unknown event flag regions in Elden Ring save files.

**Data sources:**
- 799 binary diff files (6-byte records: offset + old + new)
- JSONL metadata (player position, inventory deltas, timestamps)
- 398 proven flag positions from `ground_truth_offsets.json`
- 24,886 known flags from `extracted_event_flags.json`

**Approach:**
1. Parse all diffs, extract EF-relative single-bit changes
2. Build behavioral features per EF byte offset
3. Unsupervised clustering to find flag-like regions
4. Supervised classification using known flags as labels
5. Corroboration with inventory deltas

In [ ]:
import json
import struct
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import defaultdict, Counter
from pathlib import Path

sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (18, 6)
plt.rcParams['figure.dpi'] = 100

## 1. Load Data Sources

In [ ]:
# Paths
BASE = Path("~/dev/Elden Ring stuff")
TIMELINE_DIR = BASE / "Elden Ring save files/Granular snapshots for debugging/timeline"
DIFFS_DIR = TIMELINE_DIR / "slot_diffs"
JSONL_PATH = TIMELINE_DIR / "slot_changes.jsonl"
GROUND_TRUTH_PATH = BASE / "ER-save-Editor/ground_truth_offsets.json"
EXTRACTED_FLAGS_PATH = BASE / "ER-save-Editor/scripts/extracted_event_flags.json"

# Load JSONL metadata
metadata = []
with open(JSONL_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            metadata.append(json.loads(line))
print(f"Loaded {len(metadata)} snapshot entries")

# Load ground truth
with open(GROUND_TRUTH_PATH) as f:
    gt = json.load(f)
print(f"Ground truth: {gt['metadata']['proven_count']} proven flags")

# Build known block bases lookup
block_bases = {}
for key, info in gt.get('formulas', {}).get('block_bases', {}).items():
    block_bases[int(key)] = info
for key, info in gt.get('formulas', {}).get('midrange_formula', {}).items():
    block_bases[int(key)] = info

tile_formula = gt['formulas']['tile_formula']
dungeon_formula = {k: v for k, v in gt['formulas'].get('dungeon_formula', {}).items()
                   if k not in ('description', 'formula')}

print(f"Block bases: {len(block_bases)}")
print(f"Tile formula: base={tile_formula['base_offset']}, bps={tile_formula['bytes_per_slot']}")
print(f"Dungeon areas: {len(dungeon_formula)}")

## 2. Parse All Diffs — Extract Single-Bit Changes

In [ ]:
def parse_diff_numpy(filepath):
    """Parse binary diff into (offsets, old_vals, new_vals) arrays."""
    data = np.fromfile(filepath, dtype=np.uint8)
    if len(data) < 6:
        return np.array([], dtype=np.uint32), np.array([], dtype=np.uint8), np.array([], dtype=np.uint8)
    n = len(data) // 6
    data = data[:n * 6].reshape(n, 6)
    offsets = (data[:, 0].astype(np.uint32) |
              (data[:, 1].astype(np.uint32) << 8) |
              (data[:, 2].astype(np.uint32) << 16) |
              (data[:, 3].astype(np.uint32) << 24))
    return offsets, data[:, 4], data[:, 5]

POPCOUNT_LUT = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)

# Use numpy array accumulators (offset space is ~2M, fits in RAM easily)
MAX_OFFSET = 2_000_000  # slightly above EF total of 1.83M

total_arr = np.zeros(MAX_OFFSET, dtype=np.int32)        # total changes per offset
single_bit_arr = np.zeros(MAX_OFFSET, dtype=np.int32)   # single-bit changes
multi_bit_arr = np.zeros(MAX_OFFSET, dtype=np.int32)    # multi-bit changes
set_arr = np.zeros(MAX_OFFSET, dtype=np.int32)           # single-bit SET
clear_arr = np.zeros(MAX_OFFSET, dtype=np.int32)         # single-bit CLEAR
snap_count_arr = np.zeros(MAX_OFFSET, dtype=np.int32)   # number of distinct snapshots

total_records = 0
total_single_bit = 0
total_multi_bit = 0

# Compact single-bit data for later analysis
single_bit_compact = []

for idx, entry in enumerate(metadata):
    diff_path = DIFFS_DIR / entry['diffFile']
    if not diff_path.exists():
        continue
    offsets, old_vals, new_vals = parse_diff_numpy(str(diff_path))
    if len(offsets) == 0:
        continue
    
    # Clip offsets to our array size
    valid = offsets < MAX_OFFSET
    offsets = offsets[valid]
    old_vals = old_vals[valid]
    new_vals = new_vals[valid]
    
    xor = (old_vals ^ new_vals).astype(np.uint8)
    popcounts = POPCOUNT_LUT[xor]
    
    total_records += len(offsets)
    
    # Total changes per offset: np.add.at is O(n) numpy
    np.add.at(total_arr, offsets, 1)
    
    # Snapshot count: increment once per unique offset per snapshot
    unique_offs = np.unique(offsets)
    np.add.at(snap_count_arr, unique_offs, 1)
    
    # Single-bit
    sb_mask = popcounts == 1
    n_sb = sb_mask.sum()
    total_single_bit += n_sb
    total_multi_bit += len(offsets) - n_sb
    
    if n_sb > 0:
        sb_offsets = offsets[sb_mask]
        sb_old = old_vals[sb_mask]
        sb_new = new_vals[sb_mask]
        sb_xor = xor[sb_mask]
        sb_is_set = (sb_new > sb_old).astype(np.uint8)
        
        np.add.at(single_bit_arr, sb_offsets, 1)
        
        set_mask = sb_is_set == 1
        if set_mask.any():
            np.add.at(set_arr, sb_offsets[set_mask], 1)
        clear_mask = ~set_mask
        if clear_mask.any():
            np.add.at(clear_arr, sb_offsets[clear_mask], 1)
        
        # Store compact for later cells
        compact = np.column_stack([
            np.full(n_sb, idx, dtype=np.uint32),
            sb_offsets,
            sb_is_set.astype(np.uint32),
            sb_xor.astype(np.uint32),
        ])
        single_bit_compact.append(compact)
    
    # Multi-bit
    mb_mask = ~sb_mask
    if mb_mask.any():
        np.add.at(multi_bit_arr, offsets[mb_mask], 1)
    
    if (idx + 1) % 100 == 0:
        print(f"  Parsed {idx + 1}/{len(metadata)} diffs... "
              f"({total_records:,} records, {total_single_bit:,} single-bit)")

single_bit_all = np.vstack(single_bit_compact) if single_bit_compact else np.empty((0, 4), dtype=np.uint32)

print(f"\nTotal change records: {total_records:,}")
print(f"Single-bit changes: {total_single_bit:,} ({100*total_single_bit/total_records:.1f}%)")
print(f"Multi-bit changes: {total_multi_bit:,} ({100*total_multi_bit/total_records:.1f}%)")
print(f"Unique offsets: {(total_arr > 0).sum():,}")
print(f"Compact single-bit array shape: {single_bit_all.shape}")

In [ ]:
# Summary of single-bit vs multi-bit (already computed above)
print(f"Single-bit changes: {total_single_bit:,} ({100*total_single_bit/total_records:.1f}%)")
print(f"Multi-bit changes:  {total_multi_bit:,} ({100*total_multi_bit/total_records:.1f}%)")
print(f"Compact single-bit array: {single_bit_all.shape[0]:,} records")
print(f"  Columns: [snap_idx, offset, is_set, xor_val]")

## 3. Offset Frequency Heatmap

Visualize which byte offsets change most frequently across all 799 snapshots,
and what fraction of changes at each offset are single-bit (flag-like) vs multi-bit (counter-like).

In [ ]:
# Build per-offset DataFrame from numpy accumulators
active_mask = total_arr > 0
active_offsets = np.where(active_mask)[0]

df_offsets = pd.DataFrame({
    'offset': active_offsets,
    'total': total_arr[active_offsets],
    'single_bit': single_bit_arr[active_offsets],
    'multi_bit': multi_bit_arr[active_offsets],
    'snapshot_count': snap_count_arr[active_offsets],
    'set_count': set_arr[active_offsets],
    'clear_count': clear_arr[active_offsets],
})
df_offsets['single_bit_ratio'] = df_offsets['single_bit'] / df_offsets['total'].clip(lower=1)
df_offsets['set_ratio'] = df_offsets['set_count'] / df_offsets['single_bit'].clip(lower=1)

print(f"Unique offsets with any change: {len(df_offsets):,}")
print(f"Offset range: {df_offsets['offset'].min()} - {df_offsets['offset'].max()}")
df_offsets.head(10)

In [ ]:
# Build known EF region annotations for overlay
ef_regions = []
for block_start, info in sorted(block_bases.items()):
    base = info.get('base_offset', 0)
    size = info.get('block_size', 1000)
    status = info.get('status', 'unknown')
    if base > 0 and status not in ('disproven',):
        ef_regions.append({
            'start': base, 'end': base + size // 8 + 1,
            'name': f'Block {block_start}', 'type': 'block', 'status': status
        })

# Tile region
tile_base = tile_formula['base_offset']
tile_end = tile_base + 40 * tile_formula['slots_per_row'] * tile_formula['bytes_per_slot']
ef_regions.append({
    'start': tile_base, 'end': tile_end,
    'name': 'Tile Flags', 'type': 'tile', 'status': 'verified'
})

# Dungeon regions
for key, info in dungeon_formula.items():
    base = info.get('base_offset', 0)
    if base > 0:
        ss = info.get('section_size', 1125)
        ef_regions.append({
            'start': base, 'end': base + 50 * ss,
            'name': f'Dungeon {key}', 'type': 'dungeon', 'status': info.get('status', 'unknown')
        })

ef_regions.sort(key=lambda r: r['start'])
print(f"Known EF regions: {len(ef_regions)}")
for r in ef_regions[:10]:
    print(f"  {r['name']}: {r['start']}-{r['end']} ({r['end']-r['start']} bytes) [{r['status']}]")
print(f"  ... ({len(ef_regions) - 10} more)" if len(ef_regions) > 10 else "")

In [ ]:
# Full offset heatmap — bucket into 256-byte blocks (vectorized)
BUCKET_SIZE = 256
EF_TOTAL = 1_833_375  # Total EF section size

max_offset = int(df_offsets['offset'].max())
n_buckets = max_offset // BUCKET_SIZE + 1

# Vectorized bucketing
buckets = (df_offsets['offset'].values // BUCKET_SIZE).astype(int)
df_offsets['bucket'] = buckets

bucket_total = df_offsets.groupby('bucket')['total'].sum().reindex(range(n_buckets), fill_value=0).values
bucket_single = df_offsets.groupby('bucket')['single_bit'].sum().reindex(range(n_buckets), fill_value=0).values

# SET-only: offsets where single_bit > 0 and clear_count == 0
set_only_mask = (df_offsets['single_bit'] > 0) & (df_offsets['clear_count'] == 0)
bucket_set_only = df_offsets[set_only_mask].groupby('bucket').size().reindex(range(n_buckets), fill_value=0).values

bucket_ratio = np.where(bucket_total > 0, bucket_single / bucket_total, 0)

fig, axes = plt.subplots(3, 1, figsize=(20, 14), sharex=True)

x = np.arange(n_buckets) * BUCKET_SIZE

# Panel 1: Total change frequency (log scale)
ax = axes[0]
ax.bar(x, np.log10(bucket_total + 1), width=BUCKET_SIZE, color='steelblue', alpha=0.7)
ax.set_ylabel('log10(changes + 1)')
ax.set_title(f'Offset Change Frequency ({BUCKET_SIZE}-byte buckets, {len(metadata)} snapshots)')

# Panel 2: Single-bit ratio — use list comprehension for colors (not numpy string array)
ax = axes[1]
colors = ['red' if r > 0.8 else ('orange' if r > 0.5 else 'gray') for r in bucket_ratio]
ax.bar(x, bucket_ratio, width=BUCKET_SIZE, color=colors, alpha=0.7)
ax.set_ylabel('Single-bit ratio')
ax.set_title('Single-Bit Change Ratio (red >80% = strong flag signal)')
ax.axhline(y=0.8, color='red', linestyle='--', alpha=0.3)

# Panel 3: SET-only offset count per bucket
ax = axes[2]
ax.bar(x, bucket_set_only, width=BUCKET_SIZE, color='darkgreen', alpha=0.7)
ax.set_ylabel('SET-only offsets')
ax.set_title('SET-Only Offsets (permanent flags - never cleared)')
ax.set_xlabel('Byte offset (EF-relative)')

# Overlay known regions on all panels
region_colors = {'block': 'blue', 'tile': 'green', 'dungeon': 'purple'}
for ax in axes:
    for region in ef_regions:
        if region['status'] in ('disproven',):
            continue
        color = region_colors.get(region['type'], 'gray')
        ax.axvspan(region['start'], region['end'], alpha=0.08, color=color)

plt.tight_layout()
plt.show()

In [ ]:
# Zoom into the first 100K bytes (block + dungeon regions)
ZOOM_MAX = 100_000
mask = df_offsets['offset'] < ZOOM_MAX
df_zoom = df_offsets[mask]

fig, axes = plt.subplots(2, 1, figsize=(20, 10), sharex=True)

# Scatter: each offset colored by single-bit ratio
ax = axes[0]
sc = ax.scatter(
    df_zoom['offset'], df_zoom['snapshot_count'],
    c=df_zoom['single_bit_ratio'], cmap='RdYlGn', s=2, alpha=0.5,
    vmin=0, vmax=1
)
plt.colorbar(sc, ax=ax, label='Single-bit ratio')
ax.set_ylabel('# snapshots with change')
ax.set_title('EF Offset Behavior (0-100K): Green=flag-like, Red=counter-like')

# Overlay known block bases
for block_start, info in sorted(block_bases.items()):
    base = info.get('base_offset', 0)
    size = info.get('block_size', 1000)
    status = info.get('status', '?')
    if 0 < base < ZOOM_MAX and status not in ('disproven',):
        color = 'blue' if status == 'verified' else 'orange'
        ax.axvspan(base, base + size // 8 + 1, alpha=0.15, color=color)
        ax.text(base, ax.get_ylim()[1] * 0.95, f'{block_start}',
                fontsize=6, rotation=90, va='top', color=color)

# Panel 2: SET vs CLEAR direction for single-bit changes
ax = axes[1]
sb_mask = df_zoom['single_bit'] > 0
df_sb = df_zoom[sb_mask]
ax.scatter(df_sb['offset'], df_sb['set_count'], s=3, alpha=0.5, color='green', label='SET')
ax.scatter(df_sb['offset'], -df_sb['clear_count'], s=3, alpha=0.5, color='red', label='CLEAR')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_ylabel('SET count / -CLEAR count')
ax.set_xlabel('Byte offset')
ax.set_title('SET vs CLEAR Direction (green up = permanent flags, red down = toggling)')
ax.legend()

plt.tight_layout()
plt.show()

## 4. Feature Engineering for ML

Build a feature vector for each EF byte offset based on its observed behavior across all diffs.

In [ ]:
# Classify each offset into known regions — vectorized with interval lookup
# Build sorted interval list for fast classification
import bisect

intervals = []  # (start, end, label)
for block_start, info in block_bases.items():
    base = info.get('base_offset', 0)
    size = info.get('block_size', 1000)
    status = info.get('status', 'unknown')
    if status in ('disproven',) or base == 0:
        continue
    intervals.append((base, base + size // 8 + 1, f'block_{block_start}'))

tb = tile_formula['base_offset']
te = tb + 40 * tile_formula['slots_per_row'] * tile_formula['bytes_per_slot']
intervals.append((tb, te, 'tile'))

for key, info in dungeon_formula.items():
    base = info.get('base_offset', 0)
    if base == 0:
        continue
    ss = info.get('section_size', 1125)
    intervals.append((base, base + 50 * ss, f'dungeon_{key}'))

intervals.sort()
iv_starts = np.array([iv[0] for iv in intervals])
iv_ends = np.array([iv[1] for iv in intervals])
iv_labels = [iv[2] for iv in intervals]

def classify_offsets_vectorized(offsets):
    """Classify array of offsets into regions."""
    results = np.full(len(offsets), -1, dtype=int)  # -1 = unknown
    for i, (start, end, label) in enumerate(intervals):
        mask = (offsets >= start) & (offsets < end)
        results[mask] = i
    return results

offset_arr = df_offsets['offset'].values
region_indices = classify_offsets_vectorized(offset_arr)
regions = np.where(region_indices >= 0,
                   [iv_labels[i] if i >= 0 else 'unknown' for i in region_indices],
                   'unknown')

df_features = df_offsets.copy()
df_features['region'] = regions
df_features['is_known_flag'] = region_indices >= 0

# Derived features
df_features['set_only'] = ((df_features['single_bit'] > 0) & (df_features['clear_count'] == 0)).astype(int)
df_features['changes_per_snapshot'] = df_features['total'] / df_features['snapshot_count'].clip(lower=1)

# Rename for consistency with later cells
df_features = df_features.rename(columns={
    'total': 'total_changes',
    'single_bit': 'single_bit_count',
    'multi_bit': 'multi_bit_count',
})

# Summary
print("Offset classification:")
known = df_features[df_features['is_known_flag']]
unknown = df_features[~df_features['is_known_flag']]
print(f"  Known region offsets: {len(known):,}")
print(f"  Unknown region offsets: {len(unknown):,}")
print(f"\nKnown regions breakdown:")
print(df_features[df_features['is_known_flag']]['region'].value_counts().head(20))

In [ ]:
# Feature distributions: known flag regions vs unknown
feature_cols = ['single_bit_ratio', 'set_ratio', 'set_only', 'snapshot_count',
                'changes_per_snapshot', 'total_changes']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, col in zip(axes.flat, feature_cols):
    for label, group, color in [
        ('Known flag', df_features[df_features['is_known_flag']], 'green'),
        ('Unknown', df_features[~df_features['is_known_flag']], 'gray')
    ]:
        vals = group[col].clip(upper=group[col].quantile(0.99))  # clip outliers
        ax.hist(vals, bins=50, alpha=0.5, color=color, label=label, density=True)
    ax.set_title(col)
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions: Known Flag Regions vs Unknown', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Unsupervised Clustering

Use DBSCAN to cluster offsets by behavioral similarity, then check which clusters
overlap with known flag regions.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN

# Features for clustering
cluster_features = ['single_bit_ratio', 'set_ratio', 'set_only',
                    'changes_per_snapshot', 'snapshot_count']

X = df_features[cluster_features].values.copy()
X_scaled = StandardScaler().fit_transform(X)

# DBSCAN: find natural groupings
db = DBSCAN(eps=0.8, min_samples=10)
df_features['cluster'] = db.fit_predict(X_scaled)

n_clusters = df_features['cluster'].nunique() - (1 if -1 in df_features['cluster'].values else 0)
print(f"DBSCAN found {n_clusters} clusters (+ noise)")
print(f"\nCluster sizes:")
print(df_features['cluster'].value_counts().head(15))

In [ ]:
# For each cluster: what fraction of its members are in known flag regions?
cluster_analysis = []
for cluster_id in sorted(df_features['cluster'].unique()):
    group = df_features[df_features['cluster'] == cluster_id]
    n_known = group['is_known_flag'].sum()
    n_total = len(group)
    known_ratio = n_known / n_total
    
    # Average features
    avg_sbr = group['single_bit_ratio'].mean()
    avg_sr = group['set_ratio'].mean()
    avg_so = group['set_only'].mean()
    avg_snaps = group['snapshot_count'].mean()
    
    cluster_analysis.append({
        'cluster': cluster_id,
        'size': n_total,
        'known_count': n_known,
        'unknown_count': n_total - n_known,
        'known_ratio': known_ratio,
        'avg_single_bit_ratio': avg_sbr,
        'avg_set_ratio': avg_sr,
        'avg_set_only': avg_so,
        'avg_snapshot_count': avg_snaps,
    })

df_clusters = pd.DataFrame(cluster_analysis).sort_values('known_ratio', ascending=False)
print("Cluster analysis (sorted by known_ratio):")
print(df_clusters.to_string(index=False))

In [ ]:
# Identify "flag-like" clusters: high single-bit ratio, high set_only, low snapshot frequency
# These are clusters whose behavior matches known flag regions
flag_like_clusters = df_clusters[
    (df_clusters['avg_single_bit_ratio'] > 0.7) &
    (df_clusters['avg_set_ratio'] > 0.5) &
    (df_clusters['cluster'] != -1)  # exclude noise
]['cluster'].tolist()

print(f"Flag-like clusters: {flag_like_clusters}")
print(f"\nUnknown offsets in flag-like clusters (candidate new flag regions):")

candidates = df_features[
    (df_features['cluster'].isin(flag_like_clusters)) &
    (~df_features['is_known_flag'])
].sort_values('offset')

print(f"  Total candidate offsets: {len(candidates)}")
print(f"  Offset range: {candidates['offset'].min()} - {candidates['offset'].max()}")

# Group candidates into contiguous ranges
candidate_offsets = sorted(candidates['offset'].values)
if candidate_offsets:
    ranges = []
    start = candidate_offsets[0]
    end = candidate_offsets[0]
    count = 1
    for o in candidate_offsets[1:]:
        if o <= end + 64:  # Allow 64-byte gaps
            end = o
            count += 1
        else:
            ranges.append((start, end, count))
            start = o
            end = o
            count = 1
    ranges.append((start, end, count))
    
    print(f"\nCandidate flag regions ({len(ranges)} contiguous ranges, top 25 by density):")
    for start, end, cnt in sorted(ranges, key=lambda r: r[2], reverse=True)[:25]:
        span = end - start + 1
        density = cnt / span if span > 0 else 0
        print(f"  {start:>7} - {end:>7} (0x{start:05X}-0x{end:05X})  "
              f"{cnt:>4} offsets, span {span:>5} bytes, density {density:.2f}")

## 6. Supervised Classification

Train a Random Forest to predict whether an offset is a flag region, using known regions as labels.
Then apply to unknown offsets to get probability scores.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score

# Prepare training data from offsets that have activity
train_features = ['single_bit_ratio', 'set_ratio', 'set_only',
                  'changes_per_snapshot', 'snapshot_count', 'total_changes',
                  'single_bit_count', 'multi_bit_count', 'set_count', 'clear_count']

# Only use offsets with enough data to be meaningful
df_train = df_features[df_features['total_changes'] >= 2].copy()

X_train = df_train[train_features].values
y_train = df_train['is_known_flag'].astype(int).values

print(f"Training set: {len(X_train)} offsets")
print(f"  Positive (known flag): {y_train.sum()} ({100*y_train.mean():.1f}%)")
print(f"  Negative (unknown):    {len(y_train) - y_train.sum()} ({100*(1-y_train.mean()):.1f}%)")

# Cross-validated Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced',
                            random_state=42, n_jobs=-1)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring='roc_auc')
print(f"\nRandom Forest 5-fold CV ROC-AUC: {scores.mean():.3f} (+/- {scores.std():.3f})")

# Fit on full data for prediction
rf.fit(X_train, y_train)

# Feature importances
importances = pd.Series(rf.feature_importances_, index=train_features).sort_values(ascending=False)
print(f"\nFeature importances:")
for feat, imp in importances.items():
    print(f"  {feat:25s} {imp:.4f}")

In [ ]:
# Predict flag probability for ALL offsets
X_all = df_features[train_features].fillna(0).values
df_features['flag_probability'] = rf.predict_proba(X_all)[:, 1]

# Show top unknown offsets ranked by flag probability
unknown_ranked = df_features[~df_features['is_known_flag']].sort_values(
    'flag_probability', ascending=False
)

print("Top 50 unknown offsets by predicted flag probability:")
print(unknown_ranked[['offset', 'flag_probability', 'single_bit_ratio', 'set_ratio',
                       'set_only', 'snapshot_count', 'total_changes']].head(50).to_string(index=False))

In [ ]:
# Visualize: offset vs flag probability, colored by known/unknown
fig, ax = plt.subplots(figsize=(20, 6))

unknown = df_features[~df_features['is_known_flag']]
known = df_features[df_features['is_known_flag']]

ax.scatter(unknown['offset'], unknown['flag_probability'],
           s=2, alpha=0.3, c='red', label='Unknown (predicted)')
ax.scatter(known['offset'], known['flag_probability'],
           s=3, alpha=0.5, c='green', label='Known flag region')

ax.axhline(y=0.5, color='black', linestyle='--', alpha=0.3, label='Decision boundary')
ax.set_xlabel('Byte offset')
ax.set_ylabel('Flag probability')
ax.set_title('Random Forest Flag Probability by Offset')
ax.legend()

# Shade known regions
for region in ef_regions:
    if region['status'] not in ('disproven',):
        ax.axvspan(region['start'], region['end'], alpha=0.05, color='green')

plt.tight_layout()
plt.show()

In [ ]:
# High-confidence unknown flag regions (probability > 0.6)
high_conf = unknown_ranked[unknown_ranked['flag_probability'] > 0.6].copy()

print(f"High-confidence unknown flag candidates (p > 0.6): {len(high_conf)} offsets")

# Group into contiguous ranges
if len(high_conf) > 0:
    hc_offsets = sorted(high_conf['offset'].values)
    ranges = []
    start = hc_offsets[0]
    end = hc_offsets[0]
    offsets_in_range = [hc_offsets[0]]
    for o in hc_offsets[1:]:
        if o <= end + 32:  # 32-byte gap tolerance
            end = o
            offsets_in_range.append(o)
        else:
            ranges.append((start, end, list(offsets_in_range)))
            start = o
            end = o
            offsets_in_range = [o]
    ranges.append((start, end, list(offsets_in_range)))
    
    print(f"\nHigh-confidence candidate regions ({len(ranges)} ranges, sorted by size):")
    for start, end, offs in sorted(ranges, key=lambda r: len(r[2]), reverse=True)[:20]:
        avg_prob = high_conf[high_conf['offset'].isin(offs)]['flag_probability'].mean()
        print(f"  {start:>7} - {end:>7} (0x{start:05X}-0x{end:05X})  "
              f"{len(offs):>4} offsets, avg_prob={avg_prob:.3f}")

## 7. Temporal Co-occurrence with Inventory Deltas

For single-bit changes in unknown regions, check if they co-occur with inventory pickups.
This provides corroboration — a flag that flips when an item is picked up is likely a pickup flag.

In [ ]:
# Build snapshot metadata lookup
snap_meta = {}
for idx, entry in enumerate(metadata):
    snap_meta[idx] = {
        'id': entry['id'],
        'timestamp': entry.get('timestamp'),
        'position': entry.get('playerPosition'),
        'inventory_added': [],
        'inventory_removed': [],
        'bytes_changed': entry.get('bytesChanged', 0),
    }
    inv = entry.get('inventoryDelta')
    if inv:
        snap_meta[idx]['inventory_added'] = inv.get('added', [])
        snap_meta[idx]['inventory_removed'] = inv.get('removed', [])

# single_bit_all columns: [snap_idx=0, offset=1, is_set=2, xor_val=3]

# Vectorized classification of offsets
sb_offsets_arr = single_bit_all[:, 1].astype(int)
sb_regions = classify_offsets_vectorized(sb_offsets_arr)
sb_is_unknown = sb_regions < 0
sb_is_set_arr = single_bit_all[:, 2] == 1

# Build per-snapshot metadata arrays
snap_bytes_arr = np.array([snap_meta.get(i, {}).get('bytes_changed', 999999)
                           for i in range(len(metadata))])
snap_has_inv_arr = np.array([bool(snap_meta.get(i, {}).get('inventory_added', []))
                              for i in range(len(metadata))])

sb_snap_idx = single_bit_all[:, 0].astype(int)

# Filter: unknown + SET + small diff + has inventory
corrob_mask = (sb_is_unknown &
               sb_is_set_arr &
               (snap_bytes_arr[sb_snap_idx] < 20000) &
               snap_has_inv_arr[sb_snap_idx])

corrob_indices = np.where(corrob_mask)[0]
print(f"Corroboration events (unknown offset SET + inventory add, small diff): {len(corrob_indices)}")

if len(corrob_indices) > 0:
    corrob_offsets = single_bit_all[corrob_indices, 1].astype(int)
    corrob_xor = single_bit_all[corrob_indices, 3].astype(int)
    corrob_snaps = single_bit_all[corrob_indices, 0].astype(int)
    corrob_bits = np.log2(corrob_xor).astype(int)
    
    print(f"Unique unknown offsets with inventory corroboration: {len(np.unique(corrob_offsets))}") 
    
    print(f"\nSample events:")
    for i in range(min(20, len(corrob_indices))):
        offset = int(corrob_offsets[i])
        bit = int(corrob_bits[i])
        snap_idx = int(corrob_snaps[i])
        meta = snap_meta[snap_idx]
        items = [f"{it['itemId']}({it['category']})" for it in meta['inventory_added']]
        pos = meta['position']
        pos_str = f"({pos['x']:.0f},{pos['y']:.0f},{pos['z']:.0f})" if pos else "N/A"
        print(f"  offset={offset:>7} bit={bit} snap={meta['id']} "
              f"items=[{', '.join(items[:3])}] pos={pos_str} diff_size={meta['bytes_changed']}")

n_corroboration_events = len(corrob_indices)

## 8. Offset Co-occurrence Matrix

Find which unknown offsets change together in the same snapshot.
Co-occurring flag-like offsets likely belong to the same block/system.

In [ ]:
# Co-occurrence analysis using single_bit_all
# Filter to unknown SET changes (reuse masks from previous cell)
unk_set_mask = sb_is_unknown & sb_is_set_arr
unk_set_snaps = single_bit_all[unk_set_mask, 0].astype(int)
unk_set_offsets = single_bit_all[unk_set_mask, 1].astype(int)

# Count unique unknown SET offsets per snapshot using pandas
unk_df = pd.DataFrame({'snap': unk_set_snaps, 'offset': unk_set_offsets})
snap_offset_counts = unk_df.groupby('snap')['offset'].nunique()
signal_snap_ids = snap_offset_counts[(snap_offset_counts >= 2) & (snap_offset_counts <= 20)].index
print(f"Snapshots with 2-20 unknown SET offsets: {len(signal_snap_ids)}")

# Filter to signal snapshots
unk_signal = unk_df[unk_df['snap'].isin(signal_snap_ids)]

# Build co-occurrence pairs
cooccurrence = Counter()
offset_snap_counts_co = Counter()

for snap_idx, group in unk_signal.groupby('snap'):
    offsets = sorted(group['offset'].unique())
    for o in offsets:
        offset_snap_counts_co[o] += 1
    for i in range(len(offsets)):
        for j in range(i + 1, len(offsets)):
            cooccurrence[(offsets[i], offsets[j])] += 1

print(f"\nTop 20 co-occurring unknown offset pairs:")
for (o1, o2), count in cooccurrence.most_common(20):
    dist = o2 - o1
    print(f"  {o1:>7} + {o2:>7} (dist={dist:>5})  co-occur in {count} snapshots")

In [ ]:
# Cluster co-occurring offsets into potential block groups
# Use a simple union-find approach: if two offsets co-occur > N times, they're in the same group

MIN_COOCCURRENCE = 3

parent = {}
def find(x):
    if x not in parent:
        parent[x] = x
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb

for (o1, o2), count in cooccurrence.items():
    if count >= MIN_COOCCURRENCE:
        union(o1, o2)

# Collect groups
groups = defaultdict(list)
for o in offset_snap_counts_co:
    groups[find(o)].append(o)

# Sort groups by size
sorted_groups = sorted(groups.values(), key=len, reverse=True)

print(f"Co-occurrence groups (min {MIN_COOCCURRENCE} co-occurrences):")
print(f"  Total groups: {len(sorted_groups)}")
print(f"  Groups with 3+ members: {sum(1 for g in sorted_groups if len(g) >= 3)}")

print(f"\nTop 15 groups:")
for i, group in enumerate(sorted_groups[:15]):
    sorted_g = sorted(group)
    span = sorted_g[-1] - sorted_g[0] + 1
    density = len(sorted_g) / span if span > 0 else 0
    
    # Check if any members have high flag probability
    probs = df_features[df_features['offset'].isin(sorted_g)]['flag_probability']
    avg_prob = probs.mean() if len(probs) > 0 else 0
    
    print(f"\n  Group {i+1}: {len(sorted_g)} offsets, "
          f"range {sorted_g[0]}-{sorted_g[-1]} (0x{sorted_g[0]:05X}-0x{sorted_g[-1]:05X}), "
          f"span={span}, density={density:.3f}, avg_flag_prob={avg_prob:.3f}")
    if len(sorted_g) <= 20:
        print(f"    Offsets: {sorted_g}")
    else:
        print(f"    First 10: {sorted_g[:10]}")
        print(f"    Last 10:  {sorted_g[-10:]}")

## 9. Unknown Region Heatmap

Focused visualization of the two large unknown EF zones:
- EF+90,000 to EF+337,375 (247 KB)
- EF+1,247,375 to EF+1,833,375 (586 KB)

In [ ]:
# Zone 1: 90K - 337K
zone1 = df_features[(df_features['offset'] >= 90000) & (df_features['offset'] < 337375)]
# Zone 2: 1.25M+
zone2 = df_features[(df_features['offset'] >= 1247375) & (df_features['offset'] < 1833375)]

print(f"Unknown Zone 1 (90K-337K): {len(zone1)} active offsets")
print(f"Unknown Zone 2 (1.25M-1.83M): {len(zone2)} active offsets")

fig, axes = plt.subplots(2, 1, figsize=(20, 10))

# Zone 1
ax = axes[0]
if len(zone1) > 0:
    sc = ax.scatter(
        zone1['offset'], zone1['flag_probability'],
        c=zone1['single_bit_ratio'], cmap='RdYlGn', s=5, alpha=0.6,
        vmin=0, vmax=1
    )
    plt.colorbar(sc, ax=ax, label='Single-bit ratio')
ax.set_title('Unknown Zone 1: EF+90K to EF+337K (between dungeon and tile regions)')
ax.set_ylabel('Flag probability (RF)')
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.3)
ax.set_xlim(90000, 337375)

# Zone 2
ax = axes[1]
if len(zone2) > 0:
    sc = ax.scatter(
        zone2['offset'], zone2['flag_probability'],
        c=zone2['single_bit_ratio'], cmap='RdYlGn', s=5, alpha=0.6,
        vmin=0, vmax=1
    )
    plt.colorbar(sc, ax=ax, label='Single-bit ratio')
ax.set_title('Unknown Zone 2: EF+1.25M to EF+1.83M (after tile region)')
ax.set_ylabel('Flag probability (RF)')
ax.set_xlabel('Byte offset')
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.3)
ax.set_xlim(1247375, 1833375)

plt.tight_layout()
plt.show()

## 10. Summary: Actionable Findings

In [ ]:
print("=" * 80)
print("ML FLAG DISCOVERY SUMMARY")
print("=" * 80)

print(f"\n1. DATA SCALE")
print(f"   Snapshots: {len(metadata)}")
print(f"   Total change records: {total_records:,}")
print(f"   Single-bit changes: {total_single_bit:,}")
print(f"   Unique offsets with activity: {len(df_features):,}")

print(f"\n2. CLASSIFICATION PERFORMANCE")
print(f"   Random Forest CV ROC-AUC: {scores.mean():.3f} (+/- {scores.std():.3f})")
print(f"   Top features: {', '.join(importances.head(3).index)}")

n_high_conf = len(df_features[
    (~df_features['is_known_flag']) & (df_features['flag_probability'] > 0.6)
])
print(f"\n3. CANDIDATE FLAG REGIONS")
print(f"   Unknown offsets with p(flag) > 0.6: {n_high_conf}")
print(f"   Co-occurrence groups (3+ members): {sum(1 for g in sorted_groups if len(g) >= 3)}")
print(f"   Inventory-corroborated events: {n_corroboration_events}")

print(f"\n4. UNKNOWN ZONE ACTIVITY")
zone1_active = len(zone1[zone1['flag_probability'] > 0.5])
zone2_active = len(zone2[zone2['flag_probability'] > 0.5])
print(f"   Zone 1 (90K-337K): {zone1_active} flag-like offsets of {len(zone1)} active")
print(f"   Zone 2 (1.25M+): {zone2_active} flag-like offsets of {len(zone2)} active")

print(f"\n5. NEXT STEPS")
print(f"   - Investigate top co-occurrence groups in unknown regions")
print(f"   - Cross-reference corroborated events with known item databases")
print(f"   - Hex-dump high-probability regions to look for block structure")
print(f"   - Multi-slot differential on candidate regions for final verification")
print("=" * 80)